# Broadened CG/OR DistilBERT (machine-vs-human, all 6 domains)

DistilBERT sequence classifier for the broadened, multi-domain fake-review detector
("Model A"). 1 = CG (machine-generated) · 0 = OR (human). Trained on
`fake_reviews_broadened.csv` (products/movies/restaurants/apps/books/games).

Recipe (matches the project's other DistilBERT models): layer-wise LR decay (LLRD) +
cosine schedule with warmup + gradient clipping + mixed precision (AMP on GPU) +
early stopping on **val macro-F1** (restore best) + tqdm progress + training graph.
Exports ONNX (via optimum) for thread-safe serving.

### Instructions
1. **Runtime -> GPU**.
2. Upload `data/fake_reviews_broadened.csv` to your Drive root.
3. Run all cells. Download `fake_broadened_distilbert_onnx.zip` -> extract to
   `data/fake_broadened_distilbert_onnx/`.

In [ ]:
!pip install -q -U transformers accelerate optimum[onnxruntime] onnx onnxscript

In [ ]:
import os, random, numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (DistilBertTokenizerFast, DistilBertForSequenceClassification,
                          get_cosine_schedule_with_warmup)
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm.auto import tqdm
from google.colab import drive
drive.mount('/content/drive')

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_NAME='distilbert-base-uncased'
SPLIT_CSV='/content/drive/MyDrive/fake_reviews_broadened.csv'
MODEL_DIR='fake_broadened_distilbert_model'
MAX_LEN=256; BATCH_SIZE=32; EPOCHS=7; PATIENCE=3
BASE_LR=2e-5; HEAD_LR=1e-3; LLRD_DECAY=0.9; WARMUP_FRAC=0.1
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:', DEVICE)

import pandas as pd
df=pd.read_csv(SPLIT_CSV); df['text']=df['text'].astype(str)
def split(s): g=df[df.split==s]; return g['text'].tolist(), g['label'].tolist()
X_train,y_train=split('train'); X_val,y_val=split('val'); X_test,y_test=split('test')
print(f'train={len(X_train)} val={len(X_val)} test={len(X_test)} | labels {sorted(set(y_train))}')

In [ ]:
tokenizer=DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
class ReviewDataset(Dataset):
    def __init__(self, texts, labels):
        self.enc=tokenizer(list(texts), max_length=MAX_LEN, padding='max_length',
                           truncation=True, return_tensors='pt')
        self.labels=torch.tensor(np.asarray(labels), dtype=torch.long)
    def __len__(self): return len(self.labels)
    def __getitem__(self,i):
        return {'input_ids':self.enc['input_ids'][i],'attention_mask':self.enc['attention_mask'][i],
                'labels':self.labels[i]}
train_loader=DataLoader(ReviewDataset(X_train,y_train),batch_size=BATCH_SIZE,shuffle=True,num_workers=0)
val_loader  =DataLoader(ReviewDataset(X_val,y_val),    batch_size=64,num_workers=0)
test_loader =DataLoader(ReviewDataset(X_test,y_test),  batch_size=64,num_workers=0)

In [ ]:
model=DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters())/1e6,'M')

# layer-wise LR decay (LLRD): embeddings slowest, later layers + head faster
def make_optimizer(m):
    b=m.distilbert; n=len(b.transformer.layer); g=[]
    g.append({'params':list(b.embeddings.parameters()),'lr':BASE_LR*(LLRD_DECAY**n)})
    for i,layer in enumerate(b.transformer.layer):
        g.append({'params':list(layer.parameters()),'lr':BASE_LR*(LLRD_DECAY**(n-1-i))})
    g.append({'params':list(m.pre_classifier.parameters())+list(m.classifier.parameters()),'lr':HEAD_LR})
    return AdamW(g, weight_decay=0.01)
optimizer=make_optimizer(model)
total_steps=len(train_loader)*EPOCHS
scheduler=get_cosine_schedule_with_warmup(optimizer, int(WARMUP_FRAC*total_steps), total_steps)
print('total steps:', total_steps)

In [ ]:
USE_AMP=(DEVICE.type=='cuda')
scaler=torch.amp.GradScaler('cuda', enabled=USE_AMP)

def val_probs(loader):
    model.eval(); P,Y=[],[]
    with torch.no_grad():
        for b in loader:
            if USE_AMP:
                with torch.amp.autocast('cuda'):
                    logits=model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE)).logits
            else:
                logits=model(input_ids=b['input_ids'].to(DEVICE), attention_mask=b['attention_mask'].to(DEVICE)).logits
            P.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()); Y.extend(b['labels'].numpy())
    return np.array(P), np.array(Y)

def best_thr(p,y):
    bf,bt=-1,0.5
    for t in np.arange(0.10,0.90,0.01):
        f=f1_score(y,(p>=t).astype(int),average='macro',zero_division=0)
        if f>bf: bf,bt=f,float(t)
    return bt,bf

best_f1,best_thr_v,wait=-1,0.5,0
tr_hist,val_hist=[],[]
for epoch in range(1,EPOCHS+1):
    model.train(); tot=0
    pbar=tqdm(train_loader, desc=f'epoch {epoch}/{EPOCHS}', leave=False)
    for b in pbar:
        optimizer.zero_grad()
        ids=b['input_ids'].to(DEVICE); mask=b['attention_mask'].to(DEVICE); lab=b['labels'].to(DEVICE)
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                loss=model(input_ids=ids,attention_mask=mask,labels=lab).loss
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        else:
            loss=model(input_ids=ids,attention_mask=mask,labels=lab).loss
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        scheduler.step(); tot+=loss.item(); pbar.set_postfix(loss=f'{loss.item():.3f}')
    p,y=val_probs(val_loader); thr,f1=best_thr(p,y)
    tr_hist.append(tot/len(train_loader)); val_hist.append(f1)
    print(f'epoch {epoch}: train_loss={tot/len(train_loader):.4f}  val_macroF1={f1:.4f} @thr {thr:.2f}')
    if f1>best_f1:
        best_f1,best_thr_v=f1,thr; model.save_pretrained(MODEL_DIR); tokenizer.save_pretrained(MODEL_DIR); wait=0
    else:
        wait+=1
        if wait>=PATIENCE: print('  early stopping'); break
print(f'best val macroF1={best_f1:.4f} @thr {best_thr_v:.2f} (saved to {MODEL_DIR})')

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].plot(range(1,len(tr_hist)+1),tr_hist,marker='o'); ax[0].set_title('Train loss'); ax[0].set_xlabel('epoch')
ax[1].plot(range(1,len(val_hist)+1),val_hist,marker='o',color='green'); ax[1].set_title('Val macro-F1'); ax[1].set_xlabel('epoch')
ax[1].axvline(int(np.argmax(val_hist))+1,ls='--',color='red',label='best'); ax[1].legend()
plt.tight_layout(); plt.savefig('fake_broadened_distilbert_training.png',dpi=120); plt.show()

In [ ]:
# reload best, evaluate on test at the val-tuned threshold (official result)
best_model=DistilBertForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE); best_model.eval()
P,Y=[],[]
with torch.no_grad():
    for b in test_loader:
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                logits=best_model(input_ids=b['input_ids'].to(DEVICE),attention_mask=b['attention_mask'].to(DEVICE)).logits
        else:
            logits=best_model(input_ids=b['input_ids'].to(DEVICE),attention_mask=b['attention_mask'].to(DEVICE)).logits
        P.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()); Y.extend(b['labels'].numpy())
P,Y=np.array(P),np.array(Y); pred=(P>=best_thr_v).astype(int)
print(f'threshold={best_thr_v:.2f}  test acc={accuracy_score(Y,pred):.4f}  test macroF1={f1_score(Y,pred,average="macro"):.4f}')
print(classification_report(Y,pred,target_names=['genuine(OR)','fake(CG)'],digits=4))
import json
json.dump({'threshold':float(best_thr_v)}, open('fake_broadened_distilbert_threshold.json','w'), indent=2)

In [ ]:
# Export ONNX (optimum) + bundle threshold + tokenizer -> single zip
from optimum.onnxruntime import ORTModelForSequenceClassification
import shutil, json
ort_model=ORTModelForSequenceClassification.from_pretrained(MODEL_DIR, export=True)
ort_model.save_pretrained('fake_broadened_distilbert_onnx')
tokenizer.save_pretrained('fake_broadened_distilbert_onnx')
shutil.copy('fake_broadened_distilbert_threshold.json','fake_broadened_distilbert_onnx/threshold.json')

# sanity check
import onnxruntime as ort
print('ONNX files:', [f for f in os.listdir('fake_broadened_distilbert_onnx') if f.endswith('.onnx')])
from transformers import pipeline
pipe=pipeline('text-classification', model=ORTModelForSequenceClassification.from_pretrained('fake_broadened_distilbert_onnx', provider='CPUExecutionProvider'), tokenizer=tokenizer, device=-1)
for t in ['This product offers exceptional quality and outstanding value, highly recommended.','got these for my kids, they fight over it lol holds up ok so far']:
    r=pipe(t)[0]; print(('FAKE' if r['label']=='LABEL_1' else 'GENUINE'), f"{r['score']:.0%}", '->', t[:55])

shutil.make_archive('fake_broadened_distilbert_onnx','zip','fake_broadened_distilbert_onnx')
from google.colab import files
files.download('fake_broadened_distilbert_onnx.zip')
print('done -> fake_broadened_distilbert_onnx.zip')